# Exercício Prático — Aula 08: Web Scraping com BeautifulSoup

**Análise e Ciência de Dados** (Especialização, IFRN) — Unidade 2, Aula 08

**Entrega:** este notebook **executado** (com as saídas visíveis) + os arquivos CSV/JSON gerados, compactados em um `.zip`, na tarefa *Exercício Prático — Aula 08* do Moodle. **Prazo: 24/10/2026 (23h59).**

Preencha os trechos marcados com `# TODO`. Use o notebook `Aula08_Web_Scraping_BeautifulSoup.ipynb` como referência — os padrões de código são os mesmos.
Coloque este notebook na mesma pasta da subpasta `site_resenhas/` (vem no pacote da aula).

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, json, sys, subprocess
from io import StringIO
from urllib.parse import urljoin
from pathlib import Path
import os
from urllib.robotparser import RobotFileParser

SAIDA = Path("dados_exercicio")
SAIDA.mkdir(exist_ok=True)
CABECALHOS = {"User-Agent": "SEU-NOME-AulaACD/1.0 (exercicio educacional)"}  # TODO: coloque seu nome

def baixar(url):
    resp = requests.get(url, headers=CABECALHOS, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    return BeautifulSoup(resp.text, "html.parser")

## Parte A — Estante Potiguar (site local)

1. Coloque o site no ar (célula abaixo) e raspe **todas as resenhas**, seguindo o link "Próxima" (`li.proxima a`) e esperando 1 s entre as páginas.
2. Monte um DataFrame com: `livro`, `autor`, `nota`, `leitor`, `data`, `comentario`, `util` (use `None` quando o contador não existir).

In [7]:
servidor = subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--directory", "site_resenhas"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                            )
time.sleep(1.5)
BASE = "http://localhost:8000/"

resenhas = []
url = BASE


# TODO: laço de paginação — baixar a página, extrair cada article.resenha, seguir li.proxima a

# 
robo = RobotFileParser(urljoin(BASE, "robots.txt"))
robo.read()
pausa =  robo.crawl_delay("*") or 1
for caminho in ["index.html", "ranking.html", "privado/rascunhos.html"]:
    print(f"{caminho:25s} permitido? {robo.can_fetch(CABECALHOS['User-Agent'], urljoin(BASE, caminho))}")
print("Intervalo pedido entre requisições (Crawl-delay):", robo.crawl_delay("*"), "s")



def baixar(url):
    resp = requests.get(url, headers=CABECALHOS, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    return resp


def extrair_resenhas(soup):
    registros = []
    for art in soup.select("article.resenha"):
        util = art.select_one(".util")
        registros.append({
            "id": int(art["data-id"]),
            "livro": art.select_one("h2.livro").get_text(strip=True),
            "autor": art.select_one("p.autor").get_text(strip=True),
            "nota": int(art.select_one("div.nota")["data-nota"]),
            "leitor": art.select_one("span.nome").get_text(strip=True),
            "data": art.select_one("time")["datetime"],
            "comentario": art.select_one("p.comentario").get_text(strip=True),
            "util": int(util.get_text().split()[0]) if util else None,
        })
    return registros


while url:
    extrai_pagina = BeautifulSoup(baixar(url).text, "html.parser")
    novas = extrair_resenhas(extrai_pagina)
    resenhas.extend(novas)
    print(f"{url:40s} -> {len(novas)} resenhas (acumulado: {len(resenhas)})")

    link = extrai_pagina.select_one("li.proxima a")
    url = urljoin(url, link["href"]) if link else None
    if url:
        time.sleep(pausa)

df_resenhas = pd.DataFrame(resenhas)
print("\nTotal:", len(df_resenhas), "resenhas")



df_res = pd.DataFrame(resenhas)
print(len(df_res), "resenhas")

index.html                permitido? True
ranking.html              permitido? True
privado/rascunhos.html    permitido? False
Intervalo pedido entre requisições (Crawl-delay): 1 s
http://localhost:8000/                   -> 8 resenhas (acumulado: 8)
http://localhost:8000/pagina-2.html      -> 8 resenhas (acumulado: 16)
http://localhost:8000/pagina-3.html      -> 8 resenhas (acumulado: 24)

Total: 24 resenhas
24 resenhas


3. Calcule, **por autor**, o número de resenhas e a nota média (ordenado da maior para a menor média).
4. Mostre a resenha **mais útil** (maior valor de `util`): livro, leitor e quantidade.
5. Leia a tabela `table#ranking` de `ranking.html` com `pd.read_html(..., decimal=",")` e confira se as médias por livro batem com as que você calcularia a partir das resenhas.

In [8]:
# TODO: agrupamento por autor
df_autor_Nota_media = df_res.groupby(['autor','livro'])['nota'].agg(
    total_resenhas='count',
    nota_media='mean',
    
    
).reset_index().sort_values('nota_media',ascending=False)
# TODO: resenha mais útil
df_maior_util = df_res.groupby(['autor'])['util'].agg(
    maior_util = 'max',
    quantidade = 'count',
    
).reset_index()

df_filtro_completo = pd.merge(df_maior_util,df_res,left_on=['autor','maior_util'], right_on=['autor','util'], how='left')
colunas = ["autor","maior_util","leitor","livro","quantidade"]
df_filtro_completo = df_filtro_completo[colunas]
df_filtro_completo

# TODO: tabela de ranking com pd.read_html e comparação
resp = baixar(urljoin(BASE, "ranking.html"))
soup_rank = BeautifulSoup(resp.text, "html.parser")
tabela = soup_rank.select_one("table#ranking")


df_ranking = pd.read_html(StringIO(resp.text), attrs={"id": "ranking"}, decimal=",", thousands=".")[0]


#Seta livro como index para ser uma chave de comparacao das notas.
df_nota_media = df_autor_Nota_media.set_index('livro')
df_ranking_b = df_ranking.set_index('Livro')

#cria uma lista de comparacao entre as notas e resultando em True caso as notas forem iguais ou false caso as notas sejam diferentes.
comparar = df_nota_media['nota_media'] == df_ranking_b["Nota média"]
print(comparar)





livro
O Alienista                         True
Dom Casmurro                        True
Memórias Póstumas de Brás Cubas     True
O Cortiço                           True
Triste Fim de Policarpo Quaresma    True
Iracema                             True
dtype: bool


## Parte B — Books to Scrape (site real de treino)

6. Escolha **uma categoria com pelo menos 2 páginas** (lista no enunciado da tarefa; ex.: *Mystery*, *Romance*, *Fantasy*, *Food and Drink*). Os links das categorias estão em `div.side_categories ul li ul li a` na página inicial.
7. Raspe **todos os livros da categoria**, seguindo `li.next a` até o fim e esperando 1 s entre as páginas. Para cada livro guarde: `titulo` (atributo `title`), `preco` (float, sem o £), `nota` (1 a 5, a partir da classe `star-rating`) e `link` **absoluto** (use `urljoin` — nas categorias o `href` começa com `../../../`).

In [9]:
CATEGORIA = "Historical Fiction"
# TODO: escolha a sua
NOTAS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
paginas_lidas = 0
inicio = baixar("https://books.toscrape.com/index.html")
# TODO: encontrar o link da CATEGORIA na barra lateral
livros = []

  


def extrair_livros(soup):
    livros = []
    for pod in soup.select("article.product_pod"):
        classe_nota = pod.select_one("p.star-rating")["class"][1]      # ex.: "Three"
        livros.append({
            "titulo": pod.select_one("h3 a")["title"],
            "preco_libras": float(pod.select_one("p.price_color").get_text().replace("£", "")),
            "nota": NOTAS[classe_nota],
        })
    return livros

soup_livros = BeautifulSoup(inicio.text, "html.parser")
# TODO: laço de paginação na categoria (li.next a), com time.sleep(1)
sidebar = soup_livros.find('div', class_='side_categories')

# Busca todos os links (tags <a>) dentro da sub-lista de categorias
# Pulamos o primeiro link porque ele é apenas o texto geral "Books"
category_links = sidebar.find('ul').find('ul').find_all('a')

# Exibe o nome e o link relativo de cada categoria
for category in category_links:
    name = category.text.strip()
    link = category['href']
    if name == CATEGORIA:
        print(f"Categoria: {name} | Link: {link}")
        paginas = 0 
        url = f"https://books.toscrape.com/{link}"
        while url and paginas_lidas < 5:
            soup_categoria = BeautifulSoup(baixar(url).text, "html.parser")
            livros.extend(extrair_livros(soup_categoria))
            paginas_lidas += 1
            print(soup_categoria.select_one("li.current").get_text(strip=True), "->", len(livros), "livros")
            proxima = soup_categoria.select_one("li.next a")
            url = urljoin(url, proxima["href"]) if proxima else None
            time.sleep(1)
          

df_livros = pd.DataFrame(livros)
print(len(df_livros), "livros na categoria", CATEGORIA)

df_livros.info()


Categoria: Historical Fiction | Link: catalogue/category/books/historical-fiction_4/index.html
Page 1 of 2 -> 20 livros
Page 2 of 2 -> 26 livros
26 livros na categoria Historical Fiction
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   titulo        26 non-null     object 
 1   preco_libras  26 non-null     float64
 2   nota          26 non-null     int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 756.0+ bytes


8. Calcule: total de livros, preço médio, distribuição das notas e os **3 livros mais bem avaliados** (desempate pelo **menor** preço).
9. Abra a página de detalhe de **um** desses livros e extraia a tabela `table.table-striped` em um dicionário.

In [10]:
# TODO: estatísticas da categoria
print(df_livros.groupby("nota")["preco_libras"].agg(["count", "mean"]).round(2))
print(df_livros.loc[df_livros['nota'] == 5].sort_values('preco_libras',ascending=True).head(3))


# TODO: tabela da página de detalhe de um livro
url_livro = 'https://books.toscrape.com/catalogue/a-spys-devotion-the-regency-spies-of-london-1_3/index.html'
soup_livro = BeautifulSoup(baixar(url_livro).text, "html.parser")

print("Título:", soup_livro.select_one("div.product_main h1").get_text())
print("Categoria:", soup_livro.select("ul.breadcrumb li")[2].get_text(strip=True))
info = {tr.th.get_text(strip=True): tr.td.get_text(strip=True)
        for tr in soup_livro.select("table.table-striped tr")}
for chave, valor in info.items():
    print(f"  {chave:20s} {valor}")


      count   mean
nota              
1         5  40.87
2         4  35.05
3         5  30.51
4         4  31.94
5         8  31.24
                                               titulo  preco_libras  nota
25  A Spy's Devotion (The Regency Spies of London #1)         16.97     5
20                             Between Shades of Gray         20.79     5
15                             Voyager (Outlander #3)         21.07     5
Título: A Spy's Devotion (The Regency Spies of London #1)
Categoria: Historical Fiction
  UPC                  19fec36a1dfb4c16
  Product Type         Books
  Price (excl. tax)    £16.97
  Price (incl. tax)    £16.97
  Tax                  £0.00
  Availability         In stock (1 available)
  Number of reviews    0


## Parte C — Salvar e refletir

10. Salve em `dados_exercicio/`: as resenhas (CSV) e os livros da categoria (CSV + JSON com **proveniência**: fonte, URL, data da coleta).
11. Responda na célula Markdown abaixo (3 a 5 linhas): que cuidados éticos e legais você tomou (robots.txt, identificação, intervalo entre requisições, dados pessoais/LGPD)? O que mudaria se o site **não** permitisse a coleta?

In [11]:
# TODO: salvar CSV/JSON
df_res.to_csv(SAIDA / "resenhas_estante_potiguar.csv", index=False, encoding="utf-8" )
df_livros.to_csv(SAIDA /"books_toscrape_Historical_Fiction.csv", index=False, encoding="utf-8" )

with open(SAIDA / "books_toscrape_Historical_Fiction.json", "w", encoding="utf-8") as f:
    json.dump({
        "fonte": "Books to Scrape (site fictício para praticar web scraping)",
        "url": "https://books.toscrape.com/index.html",
        "coletado_em": "2026-10-23"       
    }, f, ensure_ascii=False, indent=2)

servidor.terminate()

**Reflexão ética (escreva aqui):**

primeiro antes de realizar o web scraping é necessário verificar se o site disponibiliza uma API que uma forma mais segura e etica para analisar os dados que queremos, verificar os termos de uso das informações constatadas pelo site, caso o site não disponibilize a API, então devemos verificar através do robots.txt quais urls são de livre acesso para a extração de informações, caso permitido e seguindo as normas da LGPD, é realizada a coleta de dados utilizando o comando time.sleep() para que seu scraping não pareça com um boot ou um ataque ao site evitando bloqueios ou banimentos das requisições. 


...